<div dir="rtl">

# 03 — تدريب Baseline: Logistic Regression بثلاثة أوزان فئات

خط أساس Logistic Regression فقط (لا XGBoost في هذا النوتبوك — وفق الترتيب المطلوب: Logistic Regression
قبل XGBoost للمقارنة). ثلاثة إعدادات لأوزان الفئات:

1. بلا أوزان (`class_weight=None`).
2. `class_weight="balanced"` (موزون عكسياً بالكامل مع تكرار الفئة).
3. أوزان جذر تربيعي — تخفيف لوزن `balanced` (`sqrt(n_samples / (n_classes * count))`) لتفادي المبالغة في
   تصحيح الاختلال الشديد بين الفئات (~247x، حسب `CLAUDE.md`).

**التدريب على عيّنة طبقية (200,000 صف من train، `random_state=42`) بدل الـ1,025,602 صف كاملة** — لتسريع
هذه التجربة الأساسية، مع الحفاظ على نسب الفئات عبر `stratify`. **التقييم فقط على `validate` الكامل — `test`
لا يُحمَّل ولا يُفتح إطلاقاً في هذا النوتبوك.**

بعد جدول المقارنة الأول: ثلاثة فحوصات تسرّب/صحة على النموذج المدرَّب، ثم جدول ثانٍ يقيّم أثر إسقاط الأدلة
عشوائياً (محاكاة مقابلة سريرية مبتورة).

</div>

In [1]:
import ast
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
SAMPLE_SIZE = 200_000
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

<div dir="rtl">

## تحميل المصفوفات والتحقق الصريح من الأبعاد

قبل أي تدريب: عدد صفوف `train` يجب أن يساوي 1,025,602 (لا أكثر ولا أقل)، وعدد أعمدة كل مصفوفة يجب أن يطابق
طول `feature_names.json` تماماً — لكل من `train` و`validate`.

</div>

In [2]:
with open(DATA_PROCESSED / "feature_names.json", encoding="utf-8") as f:
    feature_names = json.load(f)

label_encoder = joblib.load(DATA_PROCESSED / "label_encoder.joblib")

X_train_full = sparse.load_npz(DATA_PROCESSED / "train_features.npz")
y_train_full = np.load(DATA_PROCESSED / "train_labels.npy")

X_validate = sparse.load_npz(DATA_PROCESSED / "validate_features.npz")
y_validate = np.load(DATA_PROCESSED / "validate_labels.npy")

assert X_train_full.shape[0] == 1_025_602, f"عدد صفوف train غير متوقع: {X_train_full.shape[0]}"
assert X_train_full.shape[1] == len(feature_names), "عدد أعمدة X_train_full لا يطابق feature_names.json"
assert X_validate.shape[1] == len(feature_names), "عدد أعمدة X_validate لا يطابق feature_names.json"
print(f"X_train_full: {X_train_full.shape}, X_validate: {X_validate.shape}")

X_train_full: (1025602, 974), X_validate: (132448, 974)


In [3]:
# تحقق إضافي من تركيبة الأعمدة الـ974: 208 ثنائي (B) + 764 one-hot/multi-hot مدمجة (C=88 + M=676) + AGE + SEX_M
plain_cols = [c for c in feature_names if "_@_" not in c and c not in ("AGE", "SEX_M")]
suffixed_cols = [c for c in feature_names if "_@_" in c]

assert len(plain_cols) == 208, f"أعمدة B غير مطابقة: {len(plain_cols)}"
assert len(suffixed_cols) == 764, f"أعمدة C+M غير مطابقة: {len(suffixed_cols)}"
assert feature_names[-2:] == ["AGE", "SEX_M"], "العمودان الأخيران يجب أن يكونا AGE ثم SEX_M"

evidence_columns = feature_names[:-2]
item_to_col_idx = {name: idx for idx, name in enumerate(evidence_columns)}

print(f"أعمدة ثنائية (B): {len(plain_cols)}")
print(f"أعمدة one-hot/multi-hot مدمجة (C+M): {len(suffixed_cols)}")
print(f"+ AGE + SEX_M = {len(feature_names)} عموداً — تطابق تام مع 02_encode.ipynb")

أعمدة ثنائية (B): 208
أعمدة one-hot/multi-hot مدمجة (C+M): 764
+ AGE + SEX_M = 974 عموداً — تطابق تام مع 02_encode.ipynb


<div dir="rtl">

## عيّنة طبقية من `train` فقط (200,000 صف)

`stratify=y_train_full` يحافظ على النسب النسبية بين الأمراض الـ49 كما هي في `train` الكامل. `validate` و`test`
غير متأثرين بهذه الخطوة إطلاقاً — العيّنة تُؤخذ من `train` فقط.

</div>

In [4]:
X_train, _, y_train, _ = train_test_split(
    X_train_full,
    y_train_full,
    train_size=SAMPLE_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)
print(f"X_train (عيّنة طبقية): {X_train.shape}")

X_train (عيّنة طبقية): (200000, 974)


<div dir="rtl">

## خريطة severity من `release_conditions.json`

تُستخدَم فقط للتقييم (لا تدخل كميزة تدريب). `severity=1` هي فئة الخطورة القصوى في DDXPlus.

</div>

In [5]:
with open(DATA_RAW / "release_conditions.json", encoding="utf-8") as f:
    conditions = json.load(f)

severity_by_class = np.array([conditions[c]["severity"] for c in label_encoder.classes_])
severity_1_classes = set(np.where(severity_by_class == 1)[0])

print(f"عدد الأمراض severity=1: {len(severity_1_classes)} من أصل {len(label_encoder.classes_)}")
print("الأمراض:", [label_encoder.classes_[i] for i in severity_1_classes])

عدد الأمراض severity=1: 5 من أصل 49
الأمراض: ['Possible NSTEMI / STEMI', 'Acute pulmonary edema', 'Anaphylaxis', 'Ebola', 'Larygospasm']


<div dir="rtl">

## دوال المقاييس: Top-k Accuracy، ونسبة الخطأ ضمن severity=1، ودالة تقييم موحّدة

مضاف هنا **`undertriage_rate`** (المقياس الإلزامي حسب `CLAUDE.md`) و**`same_severity_confusion`**:

- **`undertriage_rate`** = نسبة الحالات التي `severity[y_pred] > severity[y_true]`. **تنويه**: الترقيم معكوس في
  DDXPlus — `severity=1` هي الأشد خطورة و`severity=5` الأخف، فـ`severity[pred] > severity[true]` يعني
  **توقّع درجة خطورة أقل من الحقيقة** (مثال: مريض NSTEMI/STEMI حقيقي `severity=1` يُصنَّف كرجفان أذيني
  `severity=3` — طارئ صُنِّف بسيطاً نسبياً). هذا هو "تحت-الفرز" الخطر فعلياً، ولا علاقة له بالضرورة بـ"نسبة
  الخطأ severity=1" أعلاه (التي تحسب أي خطأ ضمن أمراض severity=1، حتى لو كان البديل المتوقَّع severity=1
  أيضاً).
- **`same_severity_confusion`** = نسبة الحالات الخاطئة (`y_pred != y_true`) لكنها **بنفس درجة الخطورة**
  (`severity[pred] == severity[true]`) — التباس تشخيصي ضمن نفس فئة الإلحاح، وليس تراجعاً في تقدير الخطورة.

</div>

In [6]:
def top_k_accuracy(y_true, proba, k):
    top_k = np.argpartition(-proba, kth=k - 1, axis=1)[:, :k]
    match = (top_k == y_true[:, None]).any(axis=1)
    return match.mean()


def severity1_miss_rate(y_true, y_pred, severity_1_classes):
    mask = np.isin(y_true, list(severity_1_classes))
    n_severity1 = int(mask.sum())
    n_missed = int(np.sum(y_pred[mask] != y_true[mask]))
    return n_missed / n_severity1, n_severity1


def evaluate(model, X, y_true):
    proba = model.predict_proba(X)
    y_pred = model.predict(X)
    miss_rate, n_sev1 = severity1_miss_rate(y_true, y_pred, severity_1_classes)

    # severity معكوسة: رقم أكبر = أقل خطورة. undertriage = توقّع أقل خطورة من الحقيقة.
    sev_true = severity_by_class[y_true]
    sev_pred = severity_by_class[y_pred]
    undertriage_rate = float(np.mean(sev_pred > sev_true))
    same_severity_confusion = float(np.mean((y_pred != y_true) & (sev_pred == sev_true)))

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Top-3": top_k_accuracy(y_true, proba, 3),
        "Top-5": top_k_accuracy(y_true, proba, 5),
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
        "نسبة الخطأ severity=1": miss_rate,
        "عدد حالات severity=1": n_sev1,
        "undertriage_rate": undertriage_rate,
        "same_severity_confusion": same_severity_confusion,
    }

<div dir="rtl">

## أوزان الفئات الثلاثة (محسوبة على عيّنة التدريب الـ200,000، لأنها ما يُغذَّى فعلياً للنموذج)

</div>

In [7]:
classes = np.unique(y_train)
counts = np.array([(y_train == c).sum() for c in classes])
n_samples = len(y_train)
n_classes = len(classes)

balanced_weights = n_samples / (n_classes * counts)
sqrt_weights = np.sqrt(balanced_weights)

class_weight_configs = {
    "بلا أوزان": None,
    "class_weight=balanced": dict(zip(classes.tolist(), balanced_weights.tolist())),
    "أوزان جذر تربيعي": dict(zip(classes.tolist(), sqrt_weights.tolist())),
}

<div dir="rtl">

## التدريب (على العيّنة) والتقييم (على `validate` الكامل) لكل إعداد

النماذج الثلاثة تُحفَظ في `models` — سنحتاج `models["class_weight=balanced"]` لاحقاً في فحوصات التسرب
وجدول إسقاط الأدلة.

</div>

In [8]:
models = {}
results = []

for config_name, class_weight in class_weight_configs.items():
    t0 = time.time()
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight=class_weight)
    model.fit(X_train, y_train)
    fit_time = time.time() - t0
    models[config_name] = model

    metrics = evaluate(model, X_validate, y_validate)
    results.append({"الإعداد": config_name, **metrics})
    print(f"{config_name}: تم بعد {fit_time:.1f}s (Accuracy={metrics['Accuracy']:.4f}, n_iter={model.n_iter_})")

بلا أوزان: تم بعد 249.3s (Accuracy=0.9971, n_iter=[769])
class_weight=balanced: تم بعد 284.2s (Accuracy=0.9972, n_iter=[877])
أوزان جذر تربيعي: تم بعد 322.0s (Accuracy=0.9971, n_iter=[1000])


<div dir="rtl">

## جدول المقارنة

</div>

In [9]:
comparison = pd.DataFrame(results).set_index("الإعداد")
comparison

                       Accuracy  Top-3  Top-5  Macro-F1  نسبة الخطأ severity=1  عدد حالات severity=1  undertriage_rate  same_severity_confusion
الإعداد
بلا أوزان              0.997055    1.0    1.0  0.995709               0.000201                  9955          0.002227                 0.000612
class_weight=balanced  0.997176    1.0    1.0  0.996636               0.000301                  9955          0.002084                 0.000581
أوزان جذر تربيعي       0.997123    1.0    1.0  0.996173               0.000301                  9955          0.002167                 0.000596

<div dir="rtl">

## فحوصات تسرّب وصحّة (قبل الوثوق بالأرقام أعلاه)

ثلاثة فحوصات على `models["class_weight=balanced"]`:

1. هل تسرّب أي شيء من `DIFFERENTIAL_DIAGNOSIS`/`PATHOLOGY` إلى الأعمدة؟
2. هل `Top-3` محسوبة صحيحاً، وهل كل الفئات الـ49 ممثَّلة رغم التدريب على عيّنة 200 ألف فقط؟
3. هل توجد صفوف مطابقة تماماً بين `train` و`validate`؟

</div>

In [10]:
model_balanced = models["class_weight=balanced"]

# فحص 1: تسرّب من DIFFERENTIAL_DIAGNOSIS / PATHOLOGY
suspect_cols = [f for f in feature_names if "DIFF" in f.upper() or "PATHO" in f.upper()]
invalid_pattern_cols = [
    f for f in feature_names if f not in ("AGE", "SEX_M") and not f.split("_@_")[0].startswith("E_")
]

assert suspect_cols == [], f"تسرّب مشتبه به: {suspect_cols}"
assert invalid_pattern_cols == [], f"أعمدة خارج نمط E_*/AGE/SEX_M: {invalid_pattern_cols}"

print("أعمدة مشبوهة (DIFF/PATHO):", suspect_cols)
print("أعمدة خارج نمط E_*/AGE/SEX_M:", invalid_pattern_cols)
print("النتيجة: لا تسرّب — كل الأعمدة مبنية حصراً من release_evidences.json (E_*) أو AGE/SEX_M.")

أعمدة مشبوهة (DIFF/PATHO): []
أعمدة خارج نمط E_*/AGE/SEX_M: []
النتيجة: لا تسرّب — كل الأعمدة مبنية حصراً من release_evidences.json (E_*) أو AGE/SEX_M.


In [11]:
# فحص 2: صحة حساب Top-3 وتغطية الفئات الـ49
proba_val = model_balanced.predict_proba(X_validate)
print("عدد الفئات في predict_proba:", proba_val.shape[1], "| المتوقع:", len(label_encoder.classes_))
assert proba_val.shape[1] == len(label_encoder.classes_) == 49
assert set(model_balanced.classes_.tolist()) == set(range(49))
print("كل الفئات الـ49 ممثَّلة في model.classes_ رغم التدريب على عيّنة 200 ألف فقط.")

top3_impl = top_k_accuracy(y_validate, proba_val, 3)
top3_naive = np.mean([y_validate[i] in np.argsort(-proba_val[i])[:3] for i in range(len(y_validate))])
print(f"top_k_accuracy (argpartition): {top3_impl:.6f}")
print(f"تحقق ساذج (argsort، حلقة صريحة): {top3_naive:.6f}")
assert abs(top3_impl - top3_naive) < 1e-9
print("النتيجة: تطابق تام — تنفيذ argpartition صحيح.")

عدد الفئات في predict_proba: 49 | المتوقع: 49
كل الفئات الـ49 ممثَّلة في model.classes_ رغم التدريب على عيّنة 200 ألف فقط.
top_k_accuracy (argpartition): 1.000000
تحقق ساذج (argsort، حلقة صريحة): 1.000000
النتيجة: تطابق تام — تنفيذ argpartition صحيح.


<div dir="rtl">

### فحص 3: صفوف مكررة بين `train` و`validate`

بصمة رقمية سريعة لكل صف عبر `X.dot(w)` بأوزان صحيحة عشوائية (`random_state=42`) — كل القيم صحيحة صغيرة
(0/1 للأدلة، عمر ≤ ~100)، فمجموع حاصل الضرب يبقى ضمن مدى التمثيل الصحيح الدقيق لـ`float64` (أقل من 2⁵³)، أي
لا تقريب عائم إطلاقاً؛ تطابق البصمة يعني تطابقاً تاماً في القيم الصحيحة. أي تصادم بصمة يُتحقَّق منه بمقارنة
الصفين الفعليين صراحة (`!=` على المصفوفة المتناثرة، `nnz == 0` يعني تطابق تام) قبل اعتباره تكراراً حقيقياً —
وليس افتراضاً من البصمة وحدها.

</div>

In [12]:
rng_fp = np.random.default_rng(RANDOM_STATE)
w = rng_fp.integers(1, 100_000, size=X_train_full.shape[1]).astype(np.int64)

sig_train = X_train_full.dot(w)
sig_val = X_validate.dot(w)

train_sig_to_idx = {}
for i, s in enumerate(sig_train):
    train_sig_to_idx.setdefault(int(s), []).append(i)

collisions = [(j, train_sig_to_idx[int(s)]) for j, s in enumerate(sig_val) if int(s) in train_sig_to_idx]
print("صفوف validate بتصادم بصمة مع train:", len(collisions))

exact_dupe_val_rows = set()
for j, cand in collisions:
    row_val = X_validate.getrow(j)
    for i in cand:
        if (row_val != X_train_full.getrow(i)).nnz == 0:
            exact_dupe_val_rows.add(j)
            break

dupe_rate = len(exact_dupe_val_rows) / len(sig_val)
print("صفوف validate بتطابق تام مُتحقَّق منه صراحة مع صف في train:", len(exact_dupe_val_rows))
print(f"النسبة من validate: {dupe_rate:.6%}")

صفوف validate بتصادم بصمة مع train: 31032
صفوف validate بتطابق تام مُتحقَّق منه صراحة مع صف في train: 4471
النسبة من validate: 3.375664%


<div dir="rtl">

### توثيق نتائج فحوصات التسرّب الثلاثة

1. **لا تسرّب من `DIFFERENTIAL_DIAGNOSIS`/`PATHOLOGY`** — قائمة الأعمدة المشبوهة فارغة، وكل الأعمدة الـ974
   تطابق حصراً نمط `E_*` (من `release_evidences.json`) أو `AGE`/`SEX_M`. ✅
2. **`Top-3` صحيحة، وتغطية الفئات كاملة** — `predict_proba` بعرض 49 عموداً بالضبط، و`model.classes_` يغطي
   الفئات الـ49 كاملة رغم التدريب على 19.5% فقط من `train` (لم تُفقَد أي فئة نادرة بالعيّنة الطبقية).
   `top_k_accuracy` (بـ`argpartition`) يطابق تحققاً ساذجاً (`argsort` صريح) تماماً. ✅
3. **⚠️ تسرّب فعلي موجود بين splits DDXPlus الرسمية**: **4,471 من أصل 132,448 صف** في `validate`
   (**3.38%**) لها صف مطابق تماماً (كل الأعمدة الـ974) في `train` الكامل. هذا ليس خطأً في الترميز — إنه
   خاصية في DDXPlus نفسه (توليد اصطناعي قد يُنتج نفس تركيبة الأدلة+العمر+الجنس لمريضين مختلفين، خصوصاً
   الحالات البسيطة قليلة الأدلة). **الأثر العملي**: قد تكون أرقام الدقة أعلاه (~99.7%) متفائلة قليلاً بمقدار
   لا يتجاوز ~3.38% من `validate` (السقف النظري لو كانت كل هذه الصفوف مصنَّفة بالحفظ لا بالتعميم). لم نُسقِط
   هذه الصفوف من `validate` في هذا النوتبوك لأن التقسيمات الرسمية يجب أن تبقى كما هي (`لا إعادة تقسيم`)، لكن
   هذا القيد **موثَّق هنا صراحةً** ليُؤخَذ بالحسبان عند مقارنة هذا الخط الأساس بنماذج لاحقة أو بأدبيات منشورة.

</div>

<div dir="rtl">

## جدول ثانٍ: إسقاط عشوائي للأدلة (محاكاة مقابلة سريرية مبتورة)

يقيّم `models["class_weight=balanced"]` على `validate` بعد الاحتفاظ بـ`INITIAL_EVIDENCE` دائماً + k دليل
عشوائي فقط من باقي أدلة كل مريض (`AGE`/`SEX_M` غير متأثرين — ليسا "أدلة" تُجمَع تدريجياً). **الإسقاط على
مستوى الدليل (`E_204`)، لا عنصر القائمة (`E_204_@_V_7`)** — إن أُبقي دليل متعدد القيم (M)، تُبقى كل قيمه
المُفعَّلة معاً، لا قيمة واحدة منفصلة عن أخواتها. `k ∈ {3, 5, 8, 12, الكل}`، `random_state=42` لكل قيمة k
بشكل مستقل (بذرة جديدة `default_rng(42)` عند كل k، لضمان قابلية إعادة إنتاج كل عمود من الجدول بمعزل عن
ترتيب التنفيذ).

</div>

In [13]:
df_val = pd.read_csv(DATA_RAW / "release_validate_patients.csv")
df_val["EVIDENCES_PARSED"] = df_val["EVIDENCES"].apply(ast.literal_eval)


def ecode(item):
    return item.split("_@_")[0]


codes_per_row = df_val["EVIDENCES_PARSED"].apply(lambda items: sorted(set(ecode(it) for it in items)))
initial_code = df_val["INITIAL_EVIDENCE"].apply(ecode)

bad = sum(1 for ic, codes in zip(initial_code, codes_per_row) if ic not in codes)
assert bad == 0, f"{bad} صفاً INITIAL_EVIDENCE فيها ليس ضمن EVIDENCES"
print(f"تحقق: INITIAL_EVIDENCE ضمن EVIDENCES لكل الصفوف ({len(df_val) - bad:,}/{len(df_val):,}).")

تحقق: INITIAL_EVIDENCE ضمن EVIDENCES لكل الصفوف (132,448/132,448).


In [14]:
def encode_evidences(evidences_parsed, item_to_col_idx, n_evidence_cols):
    exploded = evidences_parsed.explode()
    col_idx = exploded.map(item_to_col_idx).to_numpy(dtype=np.int64)
    row_idx = exploded.index.to_numpy()
    data = np.ones(len(exploded), dtype=np.int8)
    return sparse.coo_matrix(
        (data, (row_idx, col_idx)), shape=(len(evidences_parsed), n_evidence_cols)
    ).tocsr()


age_col = X_validate[:, len(evidence_columns)]
sex_col = X_validate[:, len(evidence_columns) + 1]


def make_ablated_matrix(k, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    kept_items_per_row = []
    for items, codes, ic in zip(df_val["EVIDENCES_PARSED"], codes_per_row, initial_code):
        remaining = [c for c in codes if c != ic]
        n_take = min(k, len(remaining))
        sampled = (
            rng.choice(np.array(remaining, dtype=object), size=n_take, replace=False).tolist()
            if n_take > 0
            else []
        )
        keep_codes = set(sampled) | {ic}
        kept_items_per_row.append([it for it in items if ecode(it) in keep_codes])
    kept_series = pd.Series(kept_items_per_row, index=df_val.index)
    evid_matrix = encode_evidences(kept_series, item_to_col_idx, len(evidence_columns))
    X_k = sparse.hstack([evid_matrix, age_col, sex_col]).tocsr()
    return X_k, kept_series.apply(len).mean()

In [15]:
def with_general_error(k, avg_kept, metrics):
    row = {"k": k, "متوسط الأدلة المُبقاة": avg_kept}
    row["Accuracy"] = metrics["Accuracy"]
    row["Top-3"] = metrics["Top-3"]
    row["Top-5"] = metrics["Top-5"]
    row["Macro-F1"] = metrics["Macro-F1"]
    row["خطأ عام"] = 1 - metrics["Accuracy"]
    row["نسبة الخطأ severity=1"] = metrics["نسبة الخطأ severity=1"]
    row["عدد حالات severity=1"] = metrics["عدد حالات severity=1"]
    row["undertriage_rate"] = metrics["undertriage_rate"]
    row["same_severity_confusion"] = metrics["same_severity_confusion"]
    return row


ablation_results = []
for k in (3, 5, 8, 12):
    t0 = time.time()
    X_k, avg_kept = make_ablated_matrix(k)
    metrics = evaluate(model_balanced, X_k, y_validate)
    dt = time.time() - t0
    ablation_results.append(with_general_error(k, avg_kept, metrics))
    print(f"k={k}: تم بعد {dt:.1f}s (Accuracy={metrics['Accuracy']:.4f}, متوسط الأدلة المُبقاة={avg_kept:.2f})")

avg_original = codes_per_row.apply(len).mean()
metrics_all = evaluate(model_balanced, X_validate, y_validate)
ablation_results.append(with_general_error("الكل", avg_original, metrics_all))

k=3: تم بعد 3.5s (Accuracy=0.5693, متوسط الأدلة المُبقاة=4.86)
k=5: تم بعد 3.9s (Accuracy=0.7855, متوسط الأدلة المُبقاة=7.42)
k=8: تم بعد 3.7s (Accuracy=0.9273, متوسط الأدلة المُبقاة=11.09)
k=12: تم بعد 3.9s (Accuracy=0.9802, متوسط الأدلة المُبقاة=15.38)


<div dir="rtl">

## جدول المقارنة الثاني: الدقة مقابل عدد الأدلة المُتاحة

</div>

In [16]:
table2 = pd.DataFrame(ablation_results).set_index("k")
table2

      متوسط الأدلة المُبقاة  Accuracy     Top-3     Top-5  Macro-F1   خطأ عام  نسبة الخطأ severity=1  عدد حالات severity=1  undertriage_rate  same_severity_confusion
k
3                  4.862988  0.569333  0.744277  0.816811  0.555049  0.430667               0.600603                  9955          0.110995                 0.126291
5                  7.418874  0.785508  0.913543  0.947179  0.743350  0.214492               0.376996                  9955          0.062991                 0.066215
8                 11.094014  0.927300  0.984462  0.992216  0.895709  0.072700               0.173882                  9955          0.023723                 0.026977
12                15.375778  0.980234  0.998467  0.999592  0.971771  0.019766               0.047112                  9955          0.006040                 0.007112
الكل              15.607967  0.997176  1.000000  1.000000  0.996636  0.002824               0.000301                  9955          0.002084                 0.000581

<div dir="rtl">

### نسبة خطأ severity=1 إلى الخطأ العام

| k | خطأ عام | خطأ severity=1 | النسبة (severity=1 / عام) |
|---|---|---|---|
| 3 | 43.07% | 60.06% | **1.39×** |
| 5 | 21.45% | 37.70% | **1.76×** |
| 8 | 7.27% | 17.39% | **2.39×** |
| 12 | 1.98% | 4.71% | **2.38×** |
| الكل | 0.28% | 0.03% | **0.11×** |

**الملاحظة الأهم سريرياً**: عند نقص الأدلة (k=3..12)، خطأ severity=1 أعلى من الخطأ العام بمقدار **1.4× إلى
2.4×** — أي أن الأمراض الأشد خطورة (النوبة القلبية المحتملة، الوذمة الرئوية، الحساسية المفرطة، إيبولا، تشنج
الحنجرة) أصعب على النموذج تمييزاً من المتوسط عندما تكون المقابلة السريرية مبتورة، وهذا منطقي سريرياً: الحالات
الطارئة غالباً تحتاج مجموعة أدلة أكثر تحديداً (تمييزاً عن حالات مشابهة أقل خطورة) لا تظهر إلا بعد جمع عدد كافٍ
من الأدلة. **هذا النمط ينعكس تماماً عند الأدلة الكاملة** (النسبة 0.11×، أي خطأ severity=1 أقل من عُشر الخطأ
العام) — بكامل المعلومات، الأمراض الشديدة تصبح الأسهل تمييزاً (على الأرجح لأن أنماط أدلتها الكاملة مميّزة
جداً). **الأثر العملي**: أي منطق لاحق لإيقاف الأسئلة مبكراً (early stopping) يجب أن يكون أكثر تحفظاً
(يطلب أدلة أكثر) عندما يكون احتمال الحالة الطارئة قائماً — تماشياً مع القاعدة الذهبية في `CLAUDE.md`:
"القواعد ترفع درجة الخطورة، ولا تخفضها أبداً".

</div>

<div dir="rtl">

### `undertriage_rate` — المقياس الإلزامي حسب `CLAUDE.md`

على عكس "نسبة الخطأ severity=1" (التي تشمل أي خطأ ضمن أمراض severity=1، حتى التباساً مع مرض آخر بنفس
الخطورة)، `undertriage_rate` هنا **ينخفض بشكل رتيب تماماً** مع زيادة الأدلة (11.1% → 6.3% → 2.4% → 0.6% →
0.2% عند k=3,5,8,12,الكل على التوالي) — لا شذوذ، بعكس ما ظهر لاحقاً مع `expected_information_gain` في
`04_smart_selection.ipynb` عند k=8 (راجع ذلك النوتبوك للتفصيل والتفسير الكامل). هذا الاختيار العشوائي البسيط
لا يُظهر هذا النمط غير الرتيب — الظاهرة خاصة بديناميكية اختيار EIG تحديداً، لا بالتدهور العام مع نقص الأدلة.

</div>

<div dir="rtl">

## ملخص وخارج النطاق

**الجدول الأول (أوزان الفئات):**
- تحقّقنا صراحة قبل أي تدريب: 1,025,602 صف في `train`، 974 عموداً بنفس تركيبة `02_encode.ipynb`
  (208 ثنائي + 88 one-hot + 676 multi-hot + `AGE` + `SEX_M`)، ومطابقة تامة بين `X.shape[1]` وطول
  `feature_names.json` لكل من `train` و`validate`.
- التدريب تم على عيّنة طبقية 200,000 صف من `train` (`random_state=42`)؛ التقييم على `validate` الكامل فقط —
  **`test` لم يُحمَّل ولم يُفتح إطلاقاً في هذا النوتبوك**.
- الإعدادات الثلاثة متقاربة جداً (Accuracy ≈ 99.7%، Top-3=Top-5=100%) — راجع فحص التسرّب رقم 3 أدناه قبل
  تفسير هذا الرقم كدقة "حقيقية" بالكامل.

**فحوصات التسرّب:** لا تسرّب من `DIFFERENTIAL_DIAGNOSIS`/`PATHOLOGY`، وحساب `Top-k` صحيح مُتحقَّق منه ضد
تنفيذ ساذج، لكن **3.38% من `validate` (4,471 صف) لها توأم مطابق تماماً في `train`** — قيد حقيقي في تقسيمات
DDXPlus الرسمية نفسها، موثَّق هنا لا مُصلَح (لا نُعيد التقسيم).

**الجدول الثاني (إسقاط الأدلة):** الدقة تنهار بشكل حاد مع تقليل عدد الأدلة المتاحة — من 99.7% بكامل الأدلة
(~15.6 دليلاً بالمتوسط) إلى 56.9% عند الاحتفاظ بـ4 أدلة فقط (`INITIAL_EVIDENCE` + 3 عشوائية). هذا متوقَّع
ومنطقي (يثبت أن النموذج يعتمد فعلياً على الأدلة لا على حفظ الأنماط)، ويُقدِّم خط أساس كمي لقيمة "عمق
المقابلة السريرية" — مفيد لاحقاً عند تصميم منطق إيقاف الأسئلة المبكر (early stopping) في تجربة تفاعلية.

**`undertriage_rate` و`same_severity_confusion`** (مضافان لكل الجداول): `undertriage_rate` (توقّع خطورة أقل
من الحقيقة — الأخطر سريرياً) ينخفض رتيباً مع الإسقاط العشوائي في كلا الجدولين، بلا أي شذوذ — الشذوذ الذي ظهر
لاحقاً في `04_smart_selection.ipynb` (اختيار EIG عند k=8) خاص بديناميكية ذلك الاختيار تحديداً.

**لا XGBoost في هذا النوتبوك** — Logistic Regression فقط، كخط الأساس الإلزامي قبل XGBoost حسب `CLAUDE.md`.

</div>